<a href="https://colab.research.google.com/github/rujisayaseerai-creator/HW4-appliesML/blob/main/HW4_appliesML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
# =========================================
# 1) IMPORT
# =========================================
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB


train = pd.read_csv('https://raw.githubusercontent.com/rujisayaseerai-creator/HW4-appliesML/main/train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/rujisayaseerai-creator/HW4-appliesML/main/test.csv')

# =========================================
# 3) COPY TEST
# =========================================
test_original = test.copy()


# 3) DROP NOISE COLUMNS
# =========================================
noise_cols = ['ID', 'Customer_ID', 'Name', 'SSN','Occupation']
train.drop(columns=[c for c in noise_cols if c in train.columns], inplace=True)
test.drop(columns=[c for c in noise_cols if c in test.columns], inplace=True)

# =========================================
# 4) BASIC WEIRD VALUES -> NaN
# =========================================
def clean_weird_values(df):
    df = df.copy()
    weird_values = [
        '_', '__', '___', '____', '_____', '______', '_______',
        'NA', 'N/A', 'na', 'n/a', 'NULL', 'null', 'None', 'none',
        'nan', 'NaN', '', ' '
    ]
    df.replace(weird_values, np.nan, inplace=True)
    return df

train = clean_weird_values(train)
test = clean_weird_values(test)

# =========================================
for col in train.columns:
    if train[col].dtype == 'object':
        # train
        train[col] = (
            train[col]
            .astype(str)
            .str.strip()                     # ลบ space
            .str.replace(r'_$', '', regex=True)  # ลบ _ ท้าย
        )

        # test
        if col in test.columns:
            test[col] = (
                test[col]
                .astype(str)
                .str.strip()
                .str.replace(r'_$', '', regex=True)
            )

# =========================================
# 6) FEATURE ENGINEERING
# =========================================
# Month -> number
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8
}
if 'Month' in train.columns:
    train['Month'] = train['Month'].map(month_map)
if 'Month' in test.columns:
    test['Month'] = test['Month'].map(month_map)

# Type_of_Loan -> จำนวน loan types
if 'Type_of_Loan' in train.columns:
    train['Loan_Type_Count'] = train['Type_of_Loan'].astype(str).apply(
        lambda x: x.count(',') + 1 if x not in ['nan', 'None', ''] else np.nan
    )
    test['Loan_Type_Count'] = test['Type_of_Loan'].astype(str).apply(
        lambda x: x.count(',') + 1 if x not in ['nan', 'None', ''] else np.nan
    )
    train.drop(columns=['Type_of_Loan'], inplace=True)
    test.drop(columns=['Type_of_Loan'], inplace=True)

# =========================================
# 7) convert col to num
# =========================================
numeric_candidates = [
    'Age',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Num_Bank_Accounts',
    'Num_Credit_Card',
    'Interest_Rate',
    'Num_of_Loan',
    'Delay_from_due_date',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Num_Credit_Inquiries',
    'Outstanding_Debt',
    'Credit_Utilization_Ratio',
    'Credit_History_Age',
    'Monthly_Balance',
    'Amount_invested_monthly',
    'Monthly_EMI',
    'Loan_Type_Count'
]

# =========================================
# 8) CLEAN NUMERIC COLUMNS
# =========================================
def clean_numeric_column(series):
    series = series.astype(str).str.strip()

    # แปลงข้อความ null ปลอมให้เป็น NaN
    series = series.replace([
        'nan', 'NaN', 'None', 'none', 'NA', 'N/A', '', ' '
    ], np.nan)

    # ลบ comma เช่น 12,345.67 -> 12345.67
    series = series.str.replace(',', '', regex=False)

    # ลบอักขระทุกอย่างที่ไม่ใช่ตัวเลข จุด และลบเครื่องหมายลบ
    # เช่น 75455.34_ -> 75455.34
    # เช่น #$%8 -> 8
    series = series.str.replace(r'[^0-9.\-]', '', regex=True)

    # กันเคสแปลก เช่น '', '-', '.', '-.'
    series = series.replace(['', '-', '.', '-.'], np.nan)

    # ถ้ามีจุดหลายจุด ให้ pandas จัดการต่อด้วย coerce
    series = pd.to_numeric(series, errors='coerce')

    return series

for col in numeric_candidates:
    if col in train.columns:
        train[col] = clean_numeric_column(train[col])
    if col in test.columns:
        test[col] = clean_numeric_column(test[col])

# =========================================
# 9) CLEAN CATEGORICAL COLUMNS
# =========================================
def clean_text_column(series):
    series = series.astype(str).str.strip()

    # แทนค่าขยะเป็น NaN
    series = series.replace([
        'nan', 'NaN', 'None', 'none', 'NA', 'N/A', '', ' '
    ], np.nan)

    # ลบอักขระพิเศษ แต่เก็บตัวอักษร ตัวเลข space underscore และ hyphen
    series = series.apply(
        lambda x: re.sub(r'[^A-Za-z0-9_\-\s]', '', x) if pd.notnull(x) else x
    )

    # ลบ space ซ้ำ
    series = series.apply(
        lambda x: re.sub(r'\s+', ' ', x).strip() if pd.notnull(x) else x
    )

    # ถ้าลบแล้วเหลือค่าว่าง ให้เป็น NaN
    series = series.replace([''], np.nan)

    return series

for col in train.columns:
    if col not in numeric_candidates and train[col].dtype == 'object' and col != 'Credit_Score':
        train[col] = clean_text_column(train[col])
        if col in test.columns:
            test[col] = clean_text_column(test[col])

# =========================================
# 10) CREDIT_HISTORY_AGE (ถ้ายังเป็นข้อความ)
# ตัวอย่าง "22 Years and 1 Months" -> 265 เดือน
# =========================================
def convert_credit_history_age(series):
    def parse_age(x):
        if pd.isna(x):
            return np.nan
        x = str(x)
        m = re.search(r'(\d+)\s*Years?.*?(\d+)\s*Months?', x, flags=re.IGNORECASE)
        if m:
            years = int(m.group(1))
            months = int(m.group(2))
            return years * 12 + months

        m2 = re.search(r'(\d+)\s*Years?', x, flags=re.IGNORECASE)
        if m2:
            return int(m2.group(1)) * 12

        m3 = re.search(r'(\d+)\s*Months?', x, flags=re.IGNORECASE)
        if m3:
            return int(m3.group(1))

        return np.nan

    return series.apply(parse_age)

if 'Credit_History_Age' in train.columns and train['Credit_History_Age'].dtype == 'object':
    train['Credit_History_Age'] = convert_credit_history_age(train['Credit_History_Age'])
if 'Credit_History_Age' in test.columns and test['Credit_History_Age'].dtype == 'object':
    test['Credit_History_Age'] = convert_credit_history_age(test['Credit_History_Age'])

# =========================================
# 11) แยกประเภทคอลัมน์ใหม่
# =========================================
target_col = 'Credit_Score'

num_cols = train.drop(columns=[target_col], errors='ignore').select_dtypes(include=[np.number]).columns
cat_cols = train.drop(columns=[target_col], errors='ignore').select_dtypes(exclude=[np.number]).columns

# =========================================
# 12) FILL MISSING
# ใช้ค่า train เติมทั้ง train และ test
# =========================================
for col in num_cols:
    fill_value = train[col].median()
    if pd.isna(fill_value):
        fill_value = 0
    train[col] = train[col].fillna(fill_value)
    if col in test.columns:
        test[col] = test[col].fillna(fill_value)

for col in cat_cols:
    mode_series = train[col].mode(dropna=True)
    fill_value = mode_series.iloc[0] if len(mode_series) > 0 else 'Unknown'
    train[col] = train[col].fillna(fill_value)
    if col in test.columns:
        test[col] = test[col].fillna(fill_value)

# =========================================
# 13) OUTLIER CAPPING
# =========================================
for col in num_cols:
    q01 = train[col].quantile(0.01)
    q99 = train[col].quantile(0.99)
    train[col] = train[col].clip(q01, q99)
    if col in test.columns:
        test[col] = test[col].clip(q01, q99)

# =========================================
# 14) DROP DUPLICATES เฉพาะ train
# =========================================
train.drop_duplicates(inplace=True)


# =========================================
# 14) ENCODE CATEGORICAL FEATURES
# =========================================
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

    le_dict[col] = le


# =========================================
# 15) ENCODE TARGET
# =========================================
target_le = LabelEncoder()
train[target_col] = target_le.fit_transform(train[target_col].astype(str))


# =========================================
# 16) PREPARE X / y
# =========================================
X = train.drop(columns=[target_col])
y = train[target_col]

# ทำให้ test มีคอลัมน์ตรงกับ X
if target_col in test.columns:
    test = test.drop(columns=[target_col])

test = test[X.columns]


# =========================================
# 17) FINAL CHECK MISSING
# =========================================
for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        fill_value = X[col].median()
        if pd.isna(fill_value):
            fill_value = 0
        X[col] = X[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)
    else:
        X[col] = X[col].fillna('Unknown')
        test[col] = test[col].fillna('Unknown')

print("\nMissing train:", train.isnull().sum().sum())
print("Missing X:", X.isnull().sum().sum())
print("Missing test:", test.isnull().sum().sum())
print("Train shape after clean:", train.shape)
print("Test shape after clean:", test.shape)


# =========================================
# 18) TRAIN / VALIDATION SPLIT
# =========================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


# =========================================
# 19) SCALING
# =========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)


# =========================================
# 20) LOGISTIC REGRESSION
# =========================================
lr = LogisticRegression(
    max_iter=5000,
    C=2,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)

lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict(X_val_scaled)

f1_lr = f1_score(y_val, pred_lr, average='weighted')

print("F1 (Logistic Regression):", f1_lr)


# =========================================
# 21) NAIVE BAYES
# =========================================
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

pred_nb = nb.predict(X_val_scaled)
f1_nb = f1_score(y_val, pred_nb, average='weighted')

print("F1 (Naive Bayes):", f1_nb)


# =========================================
# 22) CHOOSE BEST MODEL
# =========================================
if f1_lr >= f1_nb:
    best_model = LogisticRegression(max_iter=5000, random_state=42)
    best_model.fit(X_scaled, y)
    pred_test = best_model.predict(test_scaled)
    print("\nBest model: Logistic Regression")
else:
    best_model = GaussianNB()
    best_model.fit(X_scaled, y)
    pred_test = best_model.predict(test_scaled)
    print("\nBest model: Naive Bayes")


# =========================================
# 23) DECODE PREDICTION
# =========================================
pred_label = target_le.inverse_transform(pred_test)


# =========================================
# =========================================
# 24) EXPORT RESULT (save + download)
# =========================================
test_original['Credit_Score'] = pred_label

# save ไฟล์
test_original.to_csv('result.csv', index=False)

# โหลดลงเครื่องทันที
from google.colab import files
files.download('result.csv')

print("\n✅ export และ download result.csv เรียบร้อย")
print("Rows in test_original:", len(test_original))
print("Rows in prediction:", len(pred_label))
print("Rows in result.csv:", len(test_original))


Missing train: 0
Missing X: 0
Missing test: 0
Train shape after clean: (80000, 23)
Test shape after clean: (20000, 22)
F1 (Logistic Regression): 0.6036055763067566
F1 (Naive Bayes): 0.5909932946783834

Best model: Logistic Regression


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ export และ download result.csv เรียบร้อย
Rows in test_original: 20000
Rows in prediction: 20000
Rows in result.csv: 20000


In [29]:
print("===== TRAIN INFO =====")
train.info()

print("\n===== TEST INFO =====")
test.info()

===== TRAIN INFO =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Month                     80000 non-null  int64  
 1   Age                       80000 non-null  float64
 2   Annual_Income             80000 non-null  float64
 3   Monthly_Inhand_Salary     80000 non-null  float64
 4   Num_Bank_Accounts         80000 non-null  float64
 5   Num_Credit_Card           80000 non-null  int64  
 6   Interest_Rate             80000 non-null  int64  
 7   Num_of_Loan               80000 non-null  int64  
 8   Delay_from_due_date       80000 non-null  int64  
 9   Num_of_Delayed_Payment    80000 non-null  float64
 10  Changed_Credit_Limit      80000 non-null  float64
 11  Num_Credit_Inquiries      80000 non-null  float64
 12  Credit_Mix                80000 non-null  int64  
 13  Outstanding_Debt          80000 non-nu

In [30]:
print("===== TRAIN DESCRIBE =====")
print(train.describe())

print("\n===== TEST DESCRIBE =====")
print(test.describe())

===== TRAIN DESCRIBE =====
              Month           Age  Annual_Income  Monthly_Inhand_Salary  \
count  80000.000000  80000.000000   80000.000000           80000.000000   
mean       4.491862     91.479362   56766.181224            4026.357174   
std        2.290262    454.656713   72113.983851            2950.982395   
min        1.000000     14.000000    7542.435150             552.166250   
25%        2.000000     24.000000   19433.067500            1784.883333   
50%        4.000000     33.000000   37550.740000            3086.756667   
75%        6.000000     42.000000   72896.120000            5369.685000   
max        8.000000   4058.050000  661365.730000           13667.903333   

       Num_Bank_Accounts  Num_Credit_Card  Interest_Rate   Num_of_Loan  \
count       80000.000000     80000.000000   80000.000000  80000.000000   
mean            9.792112        19.215025      54.880275     -0.419612   
std            40.341733        99.306410     309.806620     20.046963   
m

In [20]:
train.to_csv('train_cleaned.csv', index=False)
print("✅ export train_cleaned.csv เรียบร้อย")
from google.colab import files
files.download('result.csv')

✅ export train_cleaned.csv เรียบร้อย


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>